# Recipe Calorie Range Estimator

**Goal:** Given a recipe name and its ingredients, predict the calorie band of a dish.

| Band | Range |
|------|-------|
| `low` | < 300 kcal |
| `medium` | 300–600 kcal |
| `high` | 600–1000 kcal |
| `very_high` | > 1000 kcal |

**Pipeline:**
1. Load + curate Food.com recipe dataset (with calorie labels)
2. Baseline models (random, majority class)
3. Zero-shot frontier LLM via OpenRouter
4. Fine-tune `gpt-4.1-nano` on 100 examples via OpenAI
5. Evaluate & compare

In [ ]:
%pip install -q openai python-dotenv datasets tqdm

In [ ]:
# imports

import os
import ast
import json
import random
from collections import Counter
from dotenv import load_dotenv
from openai import OpenAI
from datasets import load_dataset
from tqdm.notebook import tqdm

from prompts.prompts import SYSTEM_PROMPT

In [ ]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# Four calorie bands (kcal per serving)
BAND_LABELS = ["low", "medium", "high", "very_high"]

In [ ]:
# environment

load_dotenv(override=True)

# OpenAI client — used ONLY for fine-tuning job management and fine-tuned model inference
openai = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# OpenRouter client — used for all zero-shot inference
router = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=os.environ["OPENROUTER_API_KEY"],
)

ZERO_SHOT_MODEL  = "openai/gpt-4o-mini"          # cheap + reliable via OpenRouter
FINETUNE_MODEL   = "gpt-4.1-nano-2025-04-14"      # cheapest OpenAI fine-tunable model
FINETUNE_SUFFIX  = "calorie-estimator"

## Set up helper functions

In [ ]:
def user_message(name: str, ingredients: list) -> str:
    return f"Recipe: {name}\nIngredients: {', '.join(str(i) for i in ingredients)}"


def finetune_example(name: str, ingredients: list, label: str) -> dict:
    """Single JSONL training record for fine-tuning."""
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_message(name, ingredients)},
            {"role": "assistant", "content": label},
        ]
    }


def inference_messages(name: str, ingredients: list) -> list:
    """Messages list for inference (no assistant turn)."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_message(name, ingredients)},
    ]


def calories_to_band(calories: float) -> str:
    """Map a raw calorie value to one of the four band labels."""
    if calories < 300:
        return "low"
    elif calories < 600:
        return "medium"
    elif calories < 1000:
        return "high"
    return "very_high"


def normalize_prediction(raw: str) -> str:
    """Coerce model output to a valid band label, or 'unknown'."""
    cleaned = raw.strip().lower().replace(" ", "_").replace("-", "_")
    for label in BAND_LABELS:
        if label in cleaned:
            return label
    return "unknown"

## Step 1 — Load & Prepare Data

Using the Food.com recipes dataset (`Shengtao/recipe`).  
The `nutrition` column stores `[calories, total_fat_PDV, sugar_PDV, sodium_PDV, protein_PDV, sat_fat_PDV, carbs_PDV]` — we extract index 0 (kcal per serving).

In [ ]:
ds = load_dataset("Shengtao/recipe", split="train")
df = ds.to_pandas()
print(f"Loaded {len(df):,} recipes")
print("Columns:", df.columns.tolist())
df.head(2)

In [ ]:
df['calories'].head(10)

In [ ]:
df.columns

In [ ]:
# Parse calories (index 0 of the nutrition list) and filter outliers
df = df.dropna(subset=["calories"])

# Keep realistic serving sizes (50–2000 kcal removes extreme outliers)
df = df[(df["calories"] >= 50) & (df["calories"] <= 2000)].reset_index(drop=True)

df["band"] = df["calories"].apply(calories_to_band)

print(f"Usable recipes: {len(df):,}")
print("Band distribution:")
print(df["band"].value_counts())

In [ ]:
# Build a balanced sample so no band dominates
# Cap each band at 500 rows, then shuffle

PER_BAND = 500
balanced = (
    df.groupby("band", group_keys=False)
      .apply(lambda g: g.sample(min(len(g), PER_BAND), random_state=42))
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
)

print(f"Balanced dataset: {len(balanced):,} recipes")
print(balanced["band"].value_counts())

In [ ]:
# Train / val / test split  (100 train · 50 val · 50 test)

train_data = balanced[:100]
val_data   = balanced[100:150]
test_data  = balanced[150:200]

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

# Peek at a sample
row = train_data.iloc[0]
print(f"\nSample: {row['title']}")
print(f"Calories: {row['calories']:.0f} → band: {row['band']}")
print(f"Ingredients: {row['ingredients']}")

## Step 2 — Baseline Models

In [ ]:
def evaluate(predictor, data, label="Model"):
    """Run predictor on data rows, print per-item results and return accuracy."""
    correct = 0
    for _, row in tqdm(data.iterrows(), total=len(data), desc=label):
        pred = normalize_prediction(predictor(row["title"], row["ingredients"]))
        hit  = pred == row["band"]
        correct += hit
        mark = "\033[92m✓\033[0m" if hit else "\033[91m✗\033[0m"
        print(f"{mark} {row['title'][:45]:<45} true={row['band']:<10} pred={pred}")
    acc = correct / len(data)
    print(f"\n{'='*60}")
    print(f"{label}: {correct}/{len(data)} = {acc:.1%} accuracy")
    print('='*60)
    return acc


# Baseline 1: random choice
def random_baseline(name, ingredients):
    return random.choice(BAND_LABELS)

# Baseline 2: always predict the most common class in training data
majority_class = Counter(train_data["band"]).most_common(1)[0][0]
def majority_baseline(name, ingredients):
    return majority_class

print(f"Majority class: {majority_class}")

In [ ]:
acc_random   = evaluate(random_baseline, test_data, label="Random")
acc_majority = evaluate(majority_baseline, test_data, label="Majority")

## Step 3 — Zero-shot with OpenRouter

Using `openai/gpt-4o-mini` via OpenRouter — no training, just prompting.

In [ ]:
def zero_shot(name, ingredients):
    response = router.chat.completions.create(
        model=ZERO_SHOT_MODEL,
        messages=inference_messages(name, ingredients),
        max_tokens=10,
        temperature=0,
    )
    return response.choices[0].message.content

# Quick sanity check
row = test_data.iloc[0]
print(zero_shot(row["title"], row["ingredients"]))
print(f"True band: {row['band']}")

In [ ]:
acc_zero_shot = evaluate(zero_shot, test_data, label="Zero-shot gpt-4o-mini")

## Step 4 — Fine-tune `gpt-4.1-nano`

> **Note:** Fine-tuning requires a usage-based billing account on OpenAI (not a free-tier account).  
> With 100 training examples and 1 epoch, the cost is well under $0.05.

In [ ]:
# Build JSONL files

def write_jsonl(data, path):
    with open(path, "w") as f:
        for _, row in data.iterrows():
            record = finetune_example(row["title"], row["ingredients"], row["band"])
            f.write(json.dumps(record) + "\n")
    print(f"Wrote {len(data)} records → {path}")

write_jsonl(train_data, "jsonl/finetune_train.jsonl")
write_jsonl(val_data,   "jsonl/finetune_val.jsonl")

In [ ]:
# Verify a sample record
with open("jsonl/finetune_train.jsonl") as f:
    print(f.readline())

In [ ]:
# Upload files to OpenAI

with open("jsonl/finetune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

with open("jsonl/finetune_val.jsonl", "rb") as f:
    val_file = openai.files.create(file=f, purpose="fine-tune")

print("Train file id:", train_file.id)
print("Val   file id:", val_file.id)

In [ ]:
# Submit fine-tuning job

job = openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model=FINETUNE_MODEL,
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix=FINETUNE_SUFFIX,
)
job_id = job.id
print("Job id:", job_id)
print("Status:", job.status)

In [ ]:
# Monitor — re-run this cell until status is 'succeeded'
# Also track at: https://platform.openai.com/finetune

job_status = openai.fine_tuning.jobs.retrieve(job_id)
print("Status:", job_status.status)

events = openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=5).data
for e in events:
    print(" -", e.message)

In [ ]:
# Retrieve fine-tuned model name once job succeeds

fine_tuned_model = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model
print("Fine-tuned model:", fine_tuned_model)

## Step 5 — Evaluate Fine-tuned Model

In [ ]:
# Fine-tuned model inference — must use OpenAI client (not OpenRouter)

def fine_tuned(name, ingredients):
    response = openai.chat.completions.create(
        model=fine_tuned_model,
        messages=inference_messages(name, ingredients),
        max_tokens=10,
        temperature=0,
    )
    return response.choices[0].message.content

# Sanity check
row = test_data.iloc[0]
print(fine_tuned(row["name"], row["ingredients"]))
print(f"True band: {row['band']}")

In [ ]:
acc_fine_tuned = evaluate(fine_tuned, test_data, label="Fine-tuned gpt-4.1-nano")

In [ ]:
# Summary comparison

results = {
    "Random baseline":         acc_random,
    "Majority baseline":       acc_majority,
    "Zero-shot gpt-4o-mini":   acc_zero_shot,
    "Fine-tuned gpt-4.1-nano": acc_fine_tuned,
}

print(f"\n{'Model':<30} {'Accuracy':>10}")
print("-" * 42)
for name, acc in results.items():
    print(f"{name:<30} {acc:>9.1%}")